# 02 - Neuron Segmentation Pipeline

This notebook walks through the neuron segmentation algorithm step by step, reproducing the logic from the `NeuronSegmenter` extension using standard Python libraries.

**Pipeline stages:**
1. Gaussian smoothing
2. Adaptive histogram equalization
3. Membrane detection (thresholding)
4. Distance-transform seeded watershed
5. Connected-component relabeling
6. Small-object removal and hole filling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.ndimage import gaussian_filter, distance_transform_edt, binary_fill_holes
from skimage.exposure import equalize_adapthist
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.color import label2rgb

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    from scipy.ndimage import label as ndi_label
    USE_CC3D = False

print(f"Using cc3d: {USE_CC3D}")

## Generate Synthetic EM Data

We create a synthetic volume that mimics EM tissue: bright cell interiors separated by dark membrane boundaries.

In [ ]:
np.random.seed(42)
shape = (32, 128, 128)

volume = np.random.randint(120, 220, size=shape, dtype=np.uint8)

# Dark membrane grid
membrane_width = 2
for offset in range(membrane_width):
    volume[:, offset::32, :] = np.random.randint(20, 70, size=volume[:, offset::32, :].shape).astype(np.uint8)
    volume[:, :, offset::32] = np.random.randint(20, 70, size=volume[:, :, offset::32].shape).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(volume[shape[0] // 2], cmap="gray")
axes[0].set_title("Synthetic EM Slice")
axes[1].hist(volume.ravel(), bins=64, color="steelblue", edgecolor="none")
axes[1].set_title("Intensity Histogram")
axes[1].set_xlabel("Intensity")
plt.tight_layout()
plt.show()

## Stage 1: Gaussian Smoothing

In [ ]:
sigma = 1.0
smoothed = gaussian_filter(volume.astype(np.float64), sigma=sigma)

mid_z = shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(volume[mid_z], cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(smoothed[mid_z], cmap="gray")
axes[1].set_title(f"Gaussian Smoothed (σ={sigma})")
plt.tight_layout()
plt.show()

## Stage 2: Adaptive Histogram Equalization

In [ ]:
chunk_size = max(1, min(shape) // 4)
equalized = equalize_adapthist(smoothed, kernel_size=chunk_size, clip_limit=0.03)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(smoothed[mid_z], cmap="gray")
axes[0].set_title("Smoothed")
axes[1].imshow(equalized[mid_z], cmap="gray")
axes[1].set_title("After CLAHE")
plt.tight_layout()
plt.show()

## Stage 3: Membrane Detection

In [ ]:
threshold = 128
rescaled = (equalized * 255).astype(np.uint8)
membrane_mask = rescaled < threshold

print(f"Membrane fraction: {membrane_mask.mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(equalized[mid_z], cmap="gray")
axes[0].set_title("Equalized")
axes[1].imshow(membrane_mask[mid_z], cmap="Reds")
axes[1].set_title(f"Membrane Mask (threshold={threshold})")
plt.tight_layout()
plt.show()

## Stage 4: Distance-Transform Watershed

In [ ]:
interior = ~membrane_mask
distance = distance_transform_edt(interior)

min_distance = max(5, int(0.02 * min(shape)))
coords = peak_local_max(distance, min_distance=min_distance, labels=interior.astype(int))
print(f"Found {len(coords)} seed points (min_distance={min_distance})")

markers = np.zeros_like(membrane_mask, dtype=np.int32)
for i, coord in enumerate(coords, start=1):
    markers[tuple(coord)] = i

labels = watershed(-distance, markers, mask=interior)
print(f"Watershed produced {labels.max()} segments")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(distance[mid_z], cmap="magma")
axes[0].set_title("Distance Transform")
axes[1].imshow(markers[mid_z] > 0, cmap="Reds")
axes[1].set_title(f"Seeds ({len(coords)} points)")
axes[2].imshow(label2rgb(labels[mid_z], bg_label=0))
axes[2].set_title("Watershed Labels")
plt.tight_layout()
plt.show()

## Stage 5: Connected-Component Relabeling

In [ ]:
if USE_CC3D:
    relabeled = cc3d.connected_components(labels, connectivity=6)
else:
    relabeled, _ = ndi_label(labels > 0)

n_neurons = relabeled.max()
print(f"Final neuron count after relabeling: {n_neurons}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(label2rgb(relabeled[mid_z], bg_label=0))
ax.set_title(f"Relabeled Segments ({n_neurons} neurons)")
ax.axis("off")
plt.tight_layout()
plt.show()

## Stage 6: Post-Processing (Cleanup)

In [ ]:
min_size = 200
cleaned = relabeled.copy()

unique_labels = np.unique(cleaned)
unique_labels = unique_labels[unique_labels != 0]
removed = 0
for lbl in unique_labels:
    if (cleaned == lbl).sum() < min_size:
        cleaned[cleaned == lbl] = 0
        removed += 1

print(f"Removed {removed} small segments (< {min_size} voxels)")

# Hole filling
for lbl in np.unique(cleaned):
    if lbl == 0:
        continue
    mask = cleaned == lbl
    filled = binary_fill_holes(mask)
    cleaned[filled & (cleaned == 0)] = lbl

final_count = len(np.unique(cleaned)) - 1
print(f"Final neuron count after cleanup: {final_count}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(label2rgb(relabeled[mid_z], bg_label=0))
axes[0].set_title("Before Cleanup")
axes[1].imshow(label2rgb(cleaned[mid_z], bg_label=0))
axes[1].set_title("After Cleanup")
plt.tight_layout()
plt.show()

## Summary

The segmentation pipeline successfully partitions the EM volume into individual neuron labels:

1. **Smoothing** reduces noise while preserving membrane boundaries
2. **CLAHE** normalizes contrast across the volume
3. **Thresholding** identifies dark membrane regions
4. **Watershed** partitions cell interiors using distance-seeded markers
5. **Connected components** ensure each disconnected region has a unique ID
6. **Cleanup** removes fragments and fills holes

This same logic is implemented in the `NeuronSegmenter` 3D Slicer extension for interactive use.

**Next:** See `03_morphology_analysis.ipynb` for quantitative analysis of the segmented neurons.